# Notebook 03_03 — Feature Engineering: Kernel PCA

Nonlinear manifold learning (RBF kernel). This is the most memory-hungry FE method:
`KernelPCA.transform()` computes a kernel matrix of shape `(n_samples, n_fit_samples)`.
With 3.4M training rows this would explode memory if done naively — this is likely why
the original monolithic notebook crashed.

**Fixes applied here**:
1. Fit KernelPCA on a small random subsample (`KPCA_FIT_SAMPLE_SIZE = 2,000`) of the training set.
2. Transform train/val/test **in batches** (`KPCA_TRANSFORM_BATCH = 50,000` rows at a time)
   so memory stays bounded to `batch_size × fit_size` instead of `n_samples × fit_size`.

**Results saved incrementally to** `results/fe_kernelpca_raw.csv` — resumable on crash.

In [ ]:
import sys; sys.path.insert(0, '/home/hanh/UAV_/notebook')
from common import *
from sklearn.decomposition import KernelPCA

print('Imports OK')

In [ ]:
d = load_data()
X_train, X_val, X_test = d['X_train'], d['X_val'], d['X_test']

METHOD_NAME = 'KernelPCA'
RESULTS_CSV = f'{RESULTS_DIR}fe_kernelpca_raw.csv'

KPCA_FIT_SAMPLE_SIZE  = 2_000
KPCA_TRANSFORM_BATCH  = 50_000

In [ ]:
def batched_transform(kpca, X, batch_size=KPCA_TRANSFORM_BATCH):
    n = len(X)
    out = np.empty((n, kpca.n_components), dtype=np.float32)
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        out[start:end] = kpca.transform(X[start:end])
    return out


def transform_fn(K):
    rng = np.random.default_rng(42)
    fit_idx = rng.choice(len(X_train), min(KPCA_FIT_SAMPLE_SIZE, len(X_train)), replace=False)
    X_fit = X_train[fit_idx]

    t0 = time.time()
    kpca = KernelPCA(n_components=K, kernel='rbf', random_state=42, n_jobs=N_JOBS)
    kpca.fit(X_fit)
    print(f'  [KernelPCA K={K}] fit on {len(X_fit)} samples in {time.time()-t0:.1f}s')

    t0 = time.time()
    Xtr = batched_transform(kpca, X_train)
    print(f'  [KernelPCA K={K}] train transform ({len(X_train):,} rows) in {time.time()-t0:.1f}s')
    Xva = batched_transform(kpca, X_val)
    Xte = batched_transform(kpca, X_test)

    return Xtr, Xva, Xte, K

In [ ]:
run_experiment_grid(
    method_name=METHOD_NAME,
    transform_fn=transform_fn,
    d=d,
    results_csv_path=RESULTS_CSV
)

In [ ]:
res_df = pd.read_csv(RESULTS_CSV)
print(res_df.groupby(['K', 'classifier'])[['f1', 'train_time_s', 'inference_ms']].agg(['mean', 'std']).to_string())